In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l2.5 Positional information

Attention is order-blind: by itself it sees a *set* of tokens, so "dog bites
man" and "man bites dog" would look identical. We add a position vector to each
token so order survives.

In [ ]:
from torch import nn

block, dim = 16, 8
pos_emb = nn.Embedding(block, dim)
p0 = pos_emb(torch.tensor(0))
p1 = pos_emb(torch.tensor(1))
print("position 0 and 1 get different vectors:", not torch.equal(p0, p1))

PocketLM (u4) adds these learned position vectors to the token embeddings.
u6 replaces them with RoPE, a smarter relative scheme.

In [ ]:
assert not torch.equal(p0, p1)